import pandas as pd
import requests
import datetime
import time

def get_binance_klines(symbol="BTCUSDT", interval="1h", start="2017-01-01"):# skida istorijske Bitcoin cene sa Binance API.
    """
    Download historical OHLCV data from Binance.
    interval examples: 1m, 5m, 15m, 1h, 4h, 1d
    """
    
    url = "https://api.binance.com/api/v3/klines"# endpoint za price data.
    start_ts = int(pd.Timestamp(start).timestamp() * 1000)# pretvara datum u timestamp.
    end_ts = int(time.time() * 1000)

    all_data = []

    while start_ts < end_ts:# Skida podatke u batch-evima jer API ima limit.
        params = {
            "symbol": symbol,
            "interval": interval,
            "startTime": start_ts,
            "limit": 1000  # Binance max per request
        }

        data = requests.get(url, params=params).json()

        # Stop if Binance returns empty list (end of available data)
        if not data:
            break

        all_data.extend(data)

        # Move start to last returned timestamp + 1ms
        last_time = data[-1][0]
        start_ts = last_time + 1

        # avoid hitting rate limit
        time.sleep(0.4)

    # Convert into DataFrame
    df = pd.DataFrame(all_data, columns=[
        "OpenTime", "Open", "High", "Low", "Close", "Volume",
        "CloseTime", "QuoteVolume", "Trades", "TakerBuyBase",
        "TakerBuyQuote", "Ignore"
    ])

    # Clean up
    df["OpenTime"] = pd.to_datetime(df["OpenTime"], unit="ms")
    df = df.set_index("OpenTime")

    df = df[["Open", "High", "Low", "Close", "Volume"]].astype(float)

    # Rename to match yfinance naming style
    df.index.name = "Datetime"

    return df

In [1]:
#df = get_binance_klines("BTCUSDT", interval="1h", start="2017-01-01")# skida Bitcoin cenu po satu od 2017.
#df.head()# prikaz prvih redova

import pandas as pd
df = pd.read_csv("btc_1h.csv", index_col=0, parse_dates=True)# ucitavanje podataka
df.head()

,Open,High,Low,Close,Volume
Datetime,,,,,
2017-08-17 04:00:00,4261.48,4313.62,4261.32,4308.83,47.181009
2017-08-17 05:00:00,4308.83,4328.69,4291.37,4315.32,23.234916
2017-08-17 06:00:00,4330.29,4345.45,4309.37,4324.35,7.229691
2017-08-17 07:00:00,4316.62,4349.99,4287.41,4349.99,4.443249
2017-08-17 08:00:00,4333.32,4377.85,4333.32,4360.69,0.972807


In [2]:
#!pip install ta
import ta

#return
df["return_1h"] = df["Close"].pct_change()# koliko se cena promenila u % zato što model lakše uči promene nego apsolutne cene.
df['daily_return'] = df['Close'].pct_change(24)# Dnevni prinos
#mean/std
df["rolling_mean_24h"] = df["Close"].rolling(24).mean()# prosečna cena zadnja 24h
df["rolling_std_24h"] = df["Close"].rolling(24).std()# volatilnost
#lag
df["close_lag_6h"] = df["Close"].shift(6)# cena pre 6h
df["close_lag_12h"] = df["Close"].shift(12)# cena pre 12h
df["close_lag_24h"] = df["Close"].shift(24)# cena pre 24h
df['close_lag_48h'] = df['Close'].shift(48)  # Cena pre 48h
df['close_lag_168h'] = df['Close'].shift(168)  # Cena pre 7 dana

# RSI
df['RSI_14'] = ta.momentum.RSIIndicator(df['Close'], window=14).rsi()

# MACD
macd = ta.trend.MACD(df['Close'], window_slow=26, window_fast=12, window_sign=9)
df['MACD'] = macd.macd()
df['MACD_signal'] = macd.macd_signal()  # MACD signal linija
df['MACD_hist'] = macd.macd_diff()  # MACD histogram

# Bollinger Bands
bb = ta.volatility.BollingerBands(df['Close'], window=20, window_dev=2)
df['BB_mavg'] = bb.bollinger_mavg()  # Srednja Bollinger linija
df['BB_upper'] = bb.bollinger_hband()  # Gornja Bollinger linija
df['BB_lower'] = bb.bollinger_lband()  # Donja Bollinger linija

# SMA i EMA
df['SMA_20'] = ta.trend.SMAIndicator(df['Close'], window=20).sma_indicator()
df['EMA_20'] = ta.trend.EMAIndicator(df['Close'], window=20).ema_indicator()

# ATR (Average True Range)
df['ATR_14'] = ta.volatility.AverageTrueRange(high=df['High'], low=df['Low'], close=df['Close'], window=14).average_true_range()

# ROC (Rate of Change) - Procenat promene cene u poslednjem periodu
df['ROC'] = ta.momentum.ROCIndicator(df['Close'], window=12).roc()

df["volatility_24h"] = df["return_1h"].rolling(24).std()

# za regresiju
df["price_24h"] = df["Close"].shift(-24)
df["price_48h"] = df["Close"].shift(-48)
df["price_7d"] = df["Close"].shift(-168)

df = df.dropna()# brisanje jer rolling i lag nekada stvaraju prazne vrednosti.
df.head() #ispis :)

,Open,High,Low,Close,Volume,return_1h,daily_return,rolling_mean_24h,rolling_std_24h,close_lag_6h,...,BB_upper,BB_lower,SMA_20,EMA_20,ATR_14,ROC,volatility_24h,price_24h,price_48h,price_7d
Datetime,,,,,,,,,,,,,,,,,,,,,
2017-08-24 04:00:00,4113.58,4148.19,4090.39,4113.98,32.247571,0.000097,0.007440,4145.985833,52.439849,4114.20,...,4249.760962,4067.399038,4158.5800,4123.466923,73.916662,-2.281456,0.006937,4310.20,4297.94,4591.56
2017-08-24 05:00:00,4113.98,4177.64,4113.49,4132.09,28.158769,0.004402,0.019469,4149.273750,48.709115,4114.01,...,4244.510883,4065.546117,4155.0285,4124.288168,73.219043,-0.820398,0.006779,4281.04,4295.75,4598.54
2017-08-24 06:00:00,4132.09,4177.18,4131.91,4133.42,29.921536,0.000322,0.013093,4151.499583,46.579934,4131.00,...,4233.637800,4066.959200,4150.2985,4125.157866,71.222683,-0.240143,0.006666,4302.72,4319.70,4572.99
2017-08-24 07:00:00,4153.97,4173.99,4133.41,4153.32,32.851584,0.004814,0.019250,4154.767917,43.628501,4140.91,...,4219.123982,4073.006018,4146.0650,4127.839974,69.033920,0.880481,0.006709,4329.00,4319.70,4592.16
2017-08-24 08:00:00,4153.80,4206.88,4153.32,4200.00,32.275428,0.011239,0.018429,4157.934583,44.054251,4131.92,...,4215.620914,4074.941086,4145.2810,4134.712358,67.928640,1.981352,0.006652,4332.17,4319.70,4586.51


In [3]:
# -----------------------------
# 1) TEMPORAL SPLIT
# -----------------------------


# izdvajamo target
target_col = "price_24h"# za sad 24h, 48h i 7d kasnije treba za test najboljeg mozela ili bar kada uspe 24h da dobije dobar rezultat prvo


features = [
    "Open","High","Low","Close","Volume",

    "return_1h",
    "daily_return",

    "rolling_mean_24h",
    "rolling_std_24h",

    "close_lag_6h",
    "close_lag_12h",
    "close_lag_24h",
    "close_lag_48h",
    "close_lag_168h",

    "RSI_14",
    "MACD",
    "MACD_signal",
    "MACD_hist",

    "ATR_14",
    "ROC",

    "SMA_20",
    "EMA_20",
    "BB_upper",
    "BB_lower"
]

# indeks za split (60/20/20)
n = len(df)
train_end = int(n * 0.6)
val_end = int(n * 0.8)

#najstariji podaci idu za train, onda za vel i na kraju test
train_df = df.iloc[:train_end]
val_df   = df.iloc[train_end:val_end]
test_df  = df.iloc[val_end:]

print(len(train_df), len(val_df), len(test_df))

44631 14877 14878


In [4]:
# -----------------------------
# 2) SCALING
# -----------------------------

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# FIT samo na train
train_df[features] = scaler.fit_transform(train_df[features])

# TRANSFORM na val i test
val_df[features] = scaler.transform(val_df[features])
test_df[features] = scaler.transform(test_df[features])

In [5]:
# -----------------------------
# 3) SEQUENCE CREATION
# -----------------------------

import numpy as np

def create_sequences(data, target, window=168):
    X, y = [], []
    for i in range(len(data) - window):
        X.append(data[i:i+window])
        y.append(target[i+window-1])# vec je pomereno
    return np.array(X), np.array(y)

window_size = 168

X_train, y_train = create_sequences(
    train_df[features].values,
    train_df[target_col].values,
    window_size
)

X_val, y_val = create_sequences(
    val_df[features].values,
    val_df[target_col].values,
    window_size
)

X_test, y_test = create_sequences(
    test_df[features].values,
    test_df[target_col].values,
    window_size
)

print(X_train.shape, X_val.shape, X_test.shape)

(44463, 168, 24) (14709, 168, 24) (14710, 168, 24)


In [6]:
# -----------------------------
# 4) TARGET SCALING
# -----------------------------

from sklearn.preprocessing import StandardScaler

y_scaler = StandardScaler()

# FIT samo na train target
y_train = y_scaler.fit_transform(y_train.reshape(-1, 1))

# TRANSFORM val i test
y_val = y_scaler.transform(y_val.reshape(-1, 1))
y_test = y_scaler.transform(y_test.reshape(-1, 1))

In [7]:
# -----------------------------
# 5) TORCH DATASET
# -----------------------------
#!pip install torch torchvision torchaudio

import torch
from torch.utils.data import Dataset, DataLoader

class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = TimeSeriesDataset(X_train, y_train)
val_dataset   = TimeSeriesDataset(X_val, y_val)
test_dataset  = TimeSeriesDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [8]:
# -----------------------------
# 6) PRICE LSTM MODEL
# -----------------------------

import torch.nn as nn

class PriceBranch(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.lstm1 = nn.LSTM(input_size, 128, batch_first=True)
        self.dropout1 = nn.Dropout(0.3)

        self.lstm2 = nn.LSTM(128, 64, batch_first=True)
        self.dropout2 = nn.Dropout(0.3)

        self.lstm3 = nn.LSTM(64, 32, batch_first=True)
        self.dropout3 = nn.Dropout(0.3)

        self.fc = nn.Linear(32, 1)

    def forward(self, x):

        out, _ = self.lstm1(x)
        out = self.dropout1(out)

        out, _ = self.lstm2(out)
        out = self.dropout2(out)

        out, _ = self.lstm3(out)
        out = self.dropout3(out)

        out = out[:, -1, :]# Uzima se samo poslednji izlaz iz sekvence
        out = self.fc(out)

        return out

In [9]:
# -----------------------------
# 7) TRAINING LOOP
# -----------------------------

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

#optimizer = get_optimizer(optimizer_type='adam', lr=0.001)# da moze da bira

def training_loop(optimizer_type='adam', lr=0.001, epochs=20):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = PriceBranch(input_size=len(features)).to(device)

    if optimizer_type == 'adam':
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    elif optimizer_type == 'sgd':
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    elif optimizer_type == 'rmsprop':
        optimizer = torch.optim.RMSprop(model.parameters(), lr=lr)
    else:
        raise ValueError(f"Unsupported optimizer type: {optimizer_type}")

    # Kreiraj funkciju gubitka sa težinama
    criterion = torch.nn.SmoothL1Loss()# Huber loss

    # learning rate scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

    # -----------------------------
    # |||| Early stopping setup ||||
    # -----------------------------
    best_val_loss = float('inf')
    patience = 5
    patience_counter = 0

    #epochs = 20

    for epoch in range(epochs):
        model.train()
        train_loss = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(X_batch)  # Model pogađa
            loss = criterion(outputs, y_batch)  # Koliko je model pogrešio
            loss.backward()  # Računanje gradijenata
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.8)  # Stabilizacija treninga (gradient clipping)
            optimizer.step()  # Model uči

            train_loss += loss.item()

        train_loss = train_loss / len(train_loader)

        # Validation
        model.eval()
        val_preds = []
        val_true = []
        val_loss = 0

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item()


                val_preds.extend(outputs.cpu().numpy())
                val_true.extend(y_batch.cpu().numpy())
        val_loss = val_loss / len(val_loader)

        # Pretvori u numpy array i reshape za scaler
        val_preds_np = np.array(val_preds).reshape(-1, 1)
        val_true_np = np.array(val_true).reshape(-1, 1)

        # VRATI NA ORIGINALNU SKALU (u dolarima)
        val_preds_inv = y_scaler.inverse_transform(val_preds_np)
        val_true_inv = y_scaler.inverse_transform(val_true_np)

        # score evaluacija
        mae = mean_absolute_error(val_true_inv, val_preds_inv)
        rmse = np.sqrt(mean_squared_error(val_true_inv, val_preds_inv))
        r2 = r2_score(val_true_inv, val_preds_inv)

        # MAPE
        val_true_np = np.array(val_true)
        val_preds_np = np.array(val_preds)
        mape = np.mean(np.abs((val_true_inv - val_preds_inv) / val_true_inv)) * 100
        print(f"\n\n-------------\n[ Epoch {epoch+1} ]\n-------------\n | Optimizer: {optimizer_type} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"MAE: {mae:.4f} | "
              f"RMSE: {rmse:.4f} | "
              f"MAPE: {mape:.2f}% | "
              f"R²: {r2:.4f}")
        print("Pogoci:")
        print("np.mean(val_preds):", np.mean(val_preds))
        print("np.mean(val_true):", np.mean(val_true))
        print("Target mean:", np.mean(y_train))
        print("Target std:", np.std(y_train))


        # Check for early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping...")
            break

        scheduler.step(val_loss)
        

In [43]:
training_loop(optimizer_type='adam', lr=0.0003, epochs=5)



-------------
[ Epoch 1 ]
-------------
 | Optimizer: adam | Train Loss: 0.0352 | Val Loss: 0.5319 | MAE: 14812.2197 | RMSE: 21120.8856 | MAPE: 32.88% | R²: -0.6529
Pogoci:
np.mean(val_preds): 0.102091976
np.mean(val_true): 0.9255752
Target mean: -1.2273054473691765e-16
Target std: 1.0


-------------
[ Epoch 2 ]
-------------
 | Optimizer: adam | Train Loss: 0.0381 | Val Loss: 0.4582 | MAE: 13020.4785 | RMSE: 19303.2623 | MAPE: 28.06% | R²: -0.3807
Pogoci:
np.mean(val_preds): 0.19881031
np.mean(val_true): 0.9255752
Target mean: -1.2273054473691765e-16
Target std: 1.0


-------------
[ Epoch 3 ]
-------------
 | Optimizer: adam | Train Loss: 0.0395 | Val Loss: 0.4386 | MAE: 12773.1182 | RMSE: 18615.7679 | MAPE: 27.77% | R²: -0.2841
Pogoci:
np.mean(val_preds): 0.20321639
np.mean(val_true): 0.9255752
Target mean: -1.2273054473691765e-16
Target std: 1.0


-------------
[ Epoch 4 ]
-------------
 | Optimizer: adam | Train Loss: 0.0333 | Val Loss: 0.4217 | MAE: 12685.1211 | RMSE: 17874.49

In [10]:
training_loop(optimizer_type='sgd', lr=0.0001, epochs=5)# manji learn rate za sgd



-------------
[ Epoch 1 ]
-------------
 | Optimizer: sgd | Train Loss: 0.4123 | Val Loss: 0.6496 | MAE: 17509.4453 | RMSE: 23702.8916 | MAPE: 40.02% | R²: -1.0817
Pogoci:
np.mean(val_preds): -0.08230615
np.mean(val_true): 0.9255752
Target mean: -1.2273054473691765e-16
Target std: 1.0


-------------
[ Epoch 2 ]
-------------
 | Optimizer: sgd | Train Loss: 0.3987 | Val Loss: 0.6189 | MAE: 16882.1152 | RMSE: 22969.2713 | MAPE: 38.46% | R²: -0.9549
Pogoci:
np.mean(val_preds): -0.043627717
np.mean(val_true): 0.9255752
Target mean: -1.2273054473691765e-16
Target std: 1.0


-------------
[ Epoch 3 ]
-------------
 | Optimizer: sgd | Train Loss: 0.3757 | Val Loss: 0.5750 | MAE: 15988.3848 | RMSE: 21888.3056 | MAPE: 36.28% | R²: -0.7752
Pogoci:
np.mean(val_preds): 0.008169918
np.mean(val_true): 0.9255752
Target mean: -1.2273054473691765e-16
Target std: 1.0


-------------
[ Epoch 4 ]
-------------
 | Optimizer: sgd | Train Loss: 0.3287 | Val Loss: 0.4903 | MAE: 14181.9453 | RMSE: 19801.177

In [11]:
training_loop(optimizer_type='rmsprop', lr=0.001, epochs=5)



-------------
[ Epoch 1 ]
-------------
 | Optimizer: rmsprop | Train Loss: 0.0114 | Val Loss: 0.6200 | MAE: 16801.5781 | RMSE: 23140.1573 | MAPE: 38.04% | R²: -0.9841
Pogoci:
np.mean(val_preds): -0.0252932
np.mean(val_true): 0.9255752
Target mean: -1.2273054473691765e-16
Target std: 1.0


-------------
[ Epoch 2 ]
-------------
 | Optimizer: rmsprop | Train Loss: 0.0105 | Val Loss: 0.6180 | MAE: 16773.4531 | RMSE: 23073.2018 | MAPE: 38.01% | R²: -0.9726
Pogoci:
np.mean(val_preds): -0.02362187
np.mean(val_true): 0.9255752
Target mean: -1.2273054473691765e-16
Target std: 1.0


-------------
[ Epoch 3 ]
-------------
 | Optimizer: rmsprop | Train Loss: 0.0112 | Val Loss: 0.6243 | MAE: 16911.6172 | RMSE: 23234.9654 | MAPE: 38.40% | R²: -1.0004
Pogoci:
np.mean(val_preds): -0.028410224
np.mean(val_true): 0.9255752
Target mean: -1.2273054473691765e-16
Target std: 1.0


-------------
[ Epoch 4 ]
-------------
 | Optimizer: rmsprop | Train Loss: 0.0116 | Val Loss: 0.6090 | MAE: 16658.8203 | 